In [1]:
!pip install protobuf==3.20.3 --quiet
!pip install transformers bitsandbytes accelerate --quiet

import importlib, site, sys
importlib.reload(site)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

<module 'site' (frozen)>

In [2]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import re
from tqdm.notebook import tqdm
import os
from kaggle_secrets import UserSecretsClient
from peft import PeftModel

2026-05-16 19:27:25.299182: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778959645.569965      49 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778959645.630048      49 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
model_id       = "mistralai/Mistral-7B-Instruct-v0.3"
adapter_path   = "/kaggle/input/datasets/linhtron123/finetune-mistral/final_model_best"
output_pretrained = "/kaggle/working/results_fewshot_pretrained.csv"
output_finetuned  = "/kaggle/working/results_fewshot_finetuned.csv"

allowed_tactics = [
    "persuasion", "playing the victim", "gaslighting", "evasion",
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

In [4]:
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded.")
except:
    hf_token = os.environ.get("HF_TOKEN")

test_df    = pd.read_csv('/kaggle/input/legaldata/test_split.csv')
example_df = pd.read_csv('/kaggle/input/legaldata/val_split.csv')
print(f"Test set: {len(test_df)} rows | Val set: {len(example_df)} rows")

Test set: 155 rows | Val set: 155 rows


In [5]:
few_shot_examples = []
for _, row in example_df.iterrows():
    few_shot_examples.append({
        "dialogue":      row["Dialogue"],
        "manipulative":  "Yes" if row["Manipulative"] == 1 else "No",
        "manipulator":   row["Primary Manipulator"] if pd.notna(row["Primary Manipulator"]) else "None",
        "techniques":    row["Manipulation Techniques"] if pd.notna(row["Manipulation Techniques"]) else "None",
    })
few_shot_examples = few_shot_examples[:5]
print(f"Few-shot examples prepared: {len(few_shot_examples)}")


Few-shot examples prepared: 5


In [6]:
def format_example(example):
    return (
        f"Dialogue: {example['dialogue']}\n"
        f"Manipulation Present: {example['manipulative']}\n"
        f"Primary Manipulator: {example['manipulator']}\n"
        f"Techniques: {example['techniques']}"
    )

In [7]:
def build_fewshot_prompt(tokenizer, dialogue, few_shot_examples):
    prompt = f"""You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Determine if any manipulation is present or not.
3. If manipulation is present, identify the primary manipulator.
4. If manipulation is present, identify the techniques used from the allowed list ONLY.

IMPORTANT: You can ONLY use these exact techniques (no variations or additions):
{', '.join(allowed_tactics)}

Here are some examples of how to analyze dialogues:

"""
    for example in few_shot_examples:
        prompt += format_example(example) + "\n\n"

    prompt += f"""Now analyze this new dialogue:

Dialogue: {dialogue}

Question: Is there manipulation present in this dialogue? If yes, identify the primary manipulator and techniques.

Response Format:
Manipulation Present: [Yes/No]
Primary Manipulator: [Plaintiff/Defendant/None]
Techniques: [technique1, technique2, ... / None]"""

    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)



In [15]:
def detect_manipulation(statement, pipe, tokenizer, few_shot_examples=None):
    prompt      = build_fewshot_prompt(tokenizer, statement, few_shot_examples)
    terminators = [pipe.tokenizer.eos_token_id]

    response = pipe(
        prompt,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
        eos_token_id=terminators,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    full_text        = response[0]["generated_text"]
    split_marker     = prompt.split("[/INST]")[-1].strip()
    prompt_end_index = full_text.find(split_marker) + len(split_marker)

    if prompt_end_index > len(split_marker):
        return full_text[prompt_end_index:].strip()
    return full_text.split("[/INST]")[-1].strip()

In [16]:
def process_and_save(pipe, df, tokenizer, few_shot_examples, output_path, label):
    print(f"\n{'='*60}")
    print(f"  Running inference: {label}")
    print(f"{'='*60}")
    print("Starting analysis...")

    manip_preds, manipulators, all_techs = [], [], []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=label):
        try:
            response_text = detect_manipulation(row["Dialogue"], pipe, tokenizer, few_shot_examples)
            m, p, t       = extract_analysis_result(response_text)
        except Exception as e:
            print(f"\n!!! LỖI xử lý hàng {idx}: {e}")
            m, p, t = 0, "Error", ["Error"]

        manip_preds.append(m)
        manipulators.append(p)
        all_techs.append(t)

        if (idx + 1) % 5 == 0:
            print(f"\nProcessed {idx+1}/{len(df)} dialogues")

        if (idx + 1) % 20 == 0:
            ckpt_df = df.iloc[:idx+1].copy()
            ckpt_df["Manipulative_Pred"]        = manip_preds
            ckpt_df["Primary_Manipulator_Pred"] = manipulators
            ckpt_df["Techniques_Pred"]          = pd.Series(all_techs).apply(
                lambda x: ", ".join(x) if isinstance(x, list) else "None"
            )
            ckpt_path = output_path.replace(".csv", f"_ckpt{idx+1}.csv")
            ckpt_df.to_csv(ckpt_path, index=False)
            print(f"[Checkpoint saved at row {idx+1} → {ckpt_path}]\n")

    result_df = df.copy()
    result_df["Manipulative_Pred"]        = manip_preds
    result_df["Primary_Manipulator_Pred"] = manipulators
    result_df["Techniques_Pred"]          = pd.Series(all_techs).apply(
        lambda x: ", ".join(x) if isinstance(x, list) else "None"
    )
    result_df.to_csv(output_path, index=False)
    print(f"\n[Done] Saved → {output_path}")
    return result_df

In [17]:
def extract_analysis_result(response):
    Manipulative_Pred   = 0
    primary_manipulator = "None"
    techniques          = []

    present_match = re.search(r"Manipulation Present:\s*(Yes|No)", response, re.IGNORECASE)
    if present_match and present_match.group(1).lower() == "yes":
        Manipulative_Pred = 1
    else:
        print(f"Final extracted values: Manipulative=0, Manipulator=None, Techniques=None")
        print("-" * 80)
        return 0, "None", ["None"]

    manipulator_match = re.search(r"Primary Manipulator:\s*([^\n]+)", response, re.IGNORECASE)
    if manipulator_match:
        txt = manipulator_match.group(1).strip().lower()
        if "plaintiff" in txt:
            primary_manipulator = "Plaintiff"
        elif "defendant" in txt:
            primary_manipulator = "Defendant"

    techniques_match = re.search(r"Techniques:\s*([^\n]+)", response, re.IGNORECASE)
    if techniques_match:
        techniques_text = techniques_match.group(1).strip()
        if techniques_text.lower() not in ["none", "n/a", "", "[]"]:
            allowed_lower = [a.lower() for a in allowed_tactics]
            for raw_tech in [t.strip().lower() for t in techniques_text.split(",")]:
                if raw_tech in allowed_lower:
                    techniques.append(allowed_tactics[allowed_lower.index(raw_tech)])

    techniques_list_or_none = techniques if techniques else ["None"]
    print(f"Final extracted values: Manipulative={Manipulative_Pred}, Manipulator={primary_manipulator}, Techniques={techniques_list_or_none}")
    print("-" * 80)
    return Manipulative_Pred, primary_manipulator, techniques_list_or_none

In [18]:
def evaluate(result_df, label):
    print(f"\n{'='*60}")
    print(f"  EVALUATION: {label}")
    print(f"{'='*60}")

    # Task 1
    targets_bin = result_df["Manipulative"].astype(int).tolist()
    preds_bin   = result_df["Manipulative_Pred"].astype(int).tolist()

    print("\n--- Task 1: Binary Classification ---")
    print(f"Accuracy:  {accuracy_score(targets_bin, preds_bin):.3f}")
    print(f"Precision: {precision_score(targets_bin, preds_bin, zero_division=0):.3f}")
    print(f"Recall:    {recall_score(targets_bin, preds_bin, zero_division=0):.3f}")
    print(f"F1 Score:  {f1_score(targets_bin, preds_bin, average='binary', zero_division=0):.3f}")
    print(f"Confusion Matrix:\n{confusion_matrix(targets_bin, preds_bin)}")

    # Task 2
    gt_manip   = result_df["Primary Manipulator"].fillna("None").str.strip().str.capitalize()
    pred_manip = result_df["Primary_Manipulator_Pred"].fillna("None").str.strip().str.capitalize()
    all_cats   = sorted(set(gt_manip.tolist() + pred_manip.tolist()))
    enc        = LabelEncoder()
    enc.fit(all_cats)
    tgt_enc    = enc.transform(gt_manip)
    pred_enc   = enc.transform(pred_manip)

    print("\n--- Task 2: Multi-class Classification ---")
    print(f"Label mapping: {dict(zip(enc.classes_, enc.transform(enc.classes_)))}")
    print(f"Accuracy:             {accuracy_score(tgt_enc, pred_enc):.3f}")
    print(f"Precision (Weighted): {precision_score(tgt_enc, pred_enc, average='weighted', zero_division=0):.3f}")
    print(f"Recall    (Weighted): {recall_score(tgt_enc, pred_enc, average='weighted', zero_division=0):.3f}")
    print(f"F1 Score  (Weighted): {f1_score(tgt_enc, pred_enc, average='weighted', zero_division=0):.3f}")
    print(f"Confusion Matrix:\n{confusion_matrix(tgt_enc, pred_enc, labels=enc.transform(enc.classes_))}")

    # Task 3
    def parse_tech(x):
        if pd.isna(x) or str(x).strip().lower() in ["none", "nan", ""]:
            return ["none"]
        return [t.strip().lower() for t in str(x).split(",") if t.strip()]

    gt_tech    = result_df["Manipulation Techniques"].apply(parse_tech)
    pred_tech  = result_df["Techniques_Pred"].apply(parse_tech)
    all_labels = sorted(set(
        l for lst in list(gt_tech) + list(pred_tech) for l in lst if l != "none"
    ))

    if all_labels:
        mlb    = MultiLabelBinarizer()
        mlb.fit([all_labels])
        y_true = mlb.transform(gt_tech)
        y_pred = mlb.transform(pred_tech)

        print("\n--- Task 3: Multi-label Classification ---")
        print(f"Classes: {mlb.classes_}")
        print(f"Exact Match (Accuracy): {accuracy_score(y_true, y_pred):.4f}")
        print(f"Precision (Macro):      {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
        print(f"Recall    (Macro):      {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
        print(f"F1 Score  (Macro):      {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
        print(f"Jaccard   (Samples):    {jaccard_score(y_true, y_pred, average='samples', zero_division=0):.4f}")


In [19]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# RUN 1: PRETRAINED Few-shot

In [13]:
print("\nLoading PRETRAINED model...")
tokenizer_pre              = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer_pre.pad_token    = tokenizer_pre.eos_token
tokenizer_pre.padding_side = "right"

base_model_pre = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", token=hf_token,
)
pipe_pretrained = pipeline(
    "text-generation", model=base_model_pre, tokenizer=tokenizer_pre, device_map="auto",
)
print("Pretrained pipeline ready.")


Loading PRETRAINED model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0


Pretrained pipeline ready.


In [20]:
results_pretrained = process_and_save(
    pipe_pretrained, test_df, tokenizer_pre,
    few_shot_examples, output_pretrained,
    "Pretrained LLM Few-shot"
)


  Running inference: Pretrained LLM Few-shot
Starting analysis...


Pretrained LLM Few-shot:   0%|          | 0/155 [00:00<?, ?it/s]

Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['emotional appeal', 'playing the victim', 'deflection', 'gaslighting']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------

Processed 5/155 dialogues
Final extracted values: Manipulative=1, Manipulator=Defendant, Techniques=['evasion', 'deflection', 'playing the victim', 'emo

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------

Processed 10/155 dialogues
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['emotional appeal', 'playing the victim', 'framing the narrative', 'evasion', 'deflection', 'minimization', 'dismissal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
-

In [21]:
evaluate(results_pretrained, "Pretrained LLM Few-shot")


  EVALUATION: Pretrained LLM Few-shot

--- Task 1: Binary Classification ---
Accuracy:  0.697
Precision: 1.000
Recall:    0.505
F1 Score:  0.671
Confusion Matrix:
[[60  0]
 [47 48]]

--- Task 2: Multi-class Classification ---
Label mapping: {'Defendant': 0, 'None': 1, 'Plaintiff': 2}
Accuracy:             0.587
Precision (Weighted): 0.720
Recall    (Weighted): 0.587
F1 Score  (Weighted): 0.540
Confusion Matrix:
[[16 30 17]
 [ 0 60  0]
 [ 0 17 15]]

--- Task 3: Multi-label Classification ---
Classes: ['character attack' 'deflection' 'dismissal' 'emotional appeal' 'evasion'
 'framing the narrative' 'gaslighting' 'guilt tripping' 'minimization'
 'persuasion' 'playing the victim']
Exact Match (Accuracy): 0.4000
Precision (Macro):      0.3351
Recall    (Macro):      0.2970
F1 Score  (Macro):      0.2982
Jaccard   (Samples):    0.0979


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:895: UserWarning: unknown class(es) ['none'] will be ignored
  warnings.warn(


In [22]:
del pipe_pretrained, base_model_pre
torch.cuda.empty_cache()
print("\nVRAM cleared.")


VRAM cleared.


# RUN 2: FINETUNED Few-shot

In [23]:
print("\nLoading FINETUNED model...")
tokenizer_ft              = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer_ft.pad_token    = tokenizer_ft.eos_token
tokenizer_ft.padding_side = "right"

base_model_ft = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", token=hf_token,
)
ft_model = PeftModel.from_pretrained(base_model_ft, adapter_path)
pipe_finetuned = pipeline(
    "text-generation", model=ft_model, tokenizer=tokenizer_ft, device_map="auto",
)
print("Finetuned pipeline ready.")



Loading FINETUNED model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version', 'target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Device set to use cuda:0


Finetuned pipeline ready.


In [24]:
results_finetuned = process_and_save(
    pipe_finetuned, test_df, tokenizer_ft,
    few_shot_examples, output_finetuned,
    "Finetuned LLM Few-shot"
)


  Running inference: Finetuned LLM Few-shot
Starting analysis...


Finetuned LLM Few-shot:   0%|          | 0/155 [00:00<?, ?it/s]

Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['deflection', 'emotional appeal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['gaslighting', 'emotional appeal']
--------------------------------------------------------------------------------
Final extracted values: Manipulative=0, Manipulator=None, Techniques=None
--------------------------------------------------------------------------------

Processed 5/155 dialogues
Final extracted values: Manipulative=1, Manipulator=Plaintiff, Techniques=['gaslighting', 'emotional appeal']
-----------------

In [25]:
evaluate(results_finetuned, "Finetuned LLM Few-shot")


  EVALUATION: Finetuned LLM Few-shot

--- Task 1: Binary Classification ---
Accuracy:  0.794
Precision: 0.985
Recall:    0.674
F1 Score:  0.800
Confusion Matrix:
[[59  1]
 [31 64]]

--- Task 2: Multi-class Classification ---
Label mapping: {'Defendant': 0, 'None': 1, 'Plaintiff': 2}
Accuracy:             0.613
Precision (Weighted): 0.734
Recall    (Weighted): 0.613
F1 Score  (Weighted): 0.586
Confusion Matrix:
[[20 15 28]
 [ 0 59  1]
 [ 0 16 16]]

--- Task 3: Multi-label Classification ---
Classes: ['character attack' 'deflection' 'dismissal' 'emotional appeal' 'evasion'
 'framing the narrative' 'gaslighting' 'guilt tripping' 'minimization'
 'persuasion' 'playing the victim']
Exact Match (Accuracy): 0.4065
Precision (Macro):      0.2812
Recall    (Macro):      0.1620
F1 Score  (Macro):      0.1691
Jaccard   (Samples):    0.1090


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:895: UserWarning: unknown class(es) ['none'] will be ignored
  warnings.warn(
